# Job Description Pipeline (Gemini)

This notebook converts `raw_jobs.csv` into a structured dataset using the Gemini API.

## Setup


In [ ]:
# Install only if needed (Colab). Comment out if already installed.
# !pip install -q pandas google-generativeai

import os
import json
import time
import re
import pandas as pd
import google.generativeai as genai


In [ ]:
# === Set your API key here ===
# Option 1: set environment variable in this cell
# os.environ["GOOGLE_API_KEY"] = "PASTE_YOUR_KEY_HERE"

# Option 2: ensure GOOGLE_API_KEY is already set in the environment
api_key = os.getenv("your API key")
if not api_key:
    raise ValueError("GOOGLE_API_KEY is not set. Please set it in the cell above.")

genai.configure(api_key=api_key)


## Load and clean raw data


In [ ]:
df = pd.read_csv("raw_jobs.csv")
print("Original rows:", len(df))

# De-duplicate by URL
df = df.drop_duplicates(subset=["url"]).copy()

# Drop rows with missing/short raw_text
df["raw_text"] = df["raw_text"].fillna("")
df = df[df["raw_text"].str.len() >= 300]

# Keep only relevant columns
keep_cols = ["job_id", "company", "title", "location", "url", "raw_text"]
df = df[keep_cols].copy()

print("Cleaned rows:", len(df))


## Create test sample


In [ ]:
sample_10 = df.sample(n=10, random_state=42) if len(df) >= 10 else df.copy()
sample_50 = df.sample(n=50, random_state=42) if len(df) >= 50 else df.copy()

print("Sample 10 rows:", len(sample_10))
print("Sample 50 rows:", len(sample_50))


## Define extraction schema and prompt


In [ ]:
ALLOWED_REMOTE = {"remote", "hybrid", "onsite", "unknown"}
ALLOWED_EMPLOYMENT = {"internship", "full_time", "part_time", "contract", "temporary", "unknown"}
ALLOWED_SENIORITY = {"intern", "entry_level", "associate", "mid_level", "senior", "lead", "manager", "director", "executive", "unknown"}

SCHEMA_FIELDS = [
    "job_id",
    "company",
    "title_raw",
    "title_clean",
    "location_raw",
    "location_clean",
    "remote_type",
    "employment_type",
    "seniority",
    "department",
    "experience_years_min",
    "experience_years_max",
    "salary_present",
    "salary_text",
    "required_skills",
    "preferred_skills",
    "responsibilities",
    "qualifications",
    "url",
]

BASE_PROMPT = """
You are an information extraction system for job descriptions.
Return ONLY valid JSON with the exact fields listed below.
Do NOT invent facts not present in the text.
If something is missing, use "unknown", null, or [] as appropriate.
Extract ONLY from the provided raw text.
Preserve these fields from the input exactly: job_id, company, title_raw, location_raw, url.

Required JSON fields:
job_id, company, title_raw, title_clean, location_raw, location_clean,
remote_type, employment_type, seniority, department,
experience_years_min, experience_years_max, salary_present, salary_text,
required_skills, preferred_skills, responsibilities, qualifications, url

Allowed categorical values:
remote_type: remote | hybrid | onsite | unknown
employment_type: internship | full_time | part_time | contract | temporary | unknown
seniority: intern | entry_level | associate | mid_level | senior | lead | manager | director | executive | unknown

Return valid JSON only.
"""


## Gemini extraction function


In [ ]:
def extract_job_structured(row, model_name="gemini-1.5-flash"):
    """
    Send one job to Gemini and parse JSON response.
    Returns a dict or raises an Exception for the caller to handle.
    """
    model = genai.GenerativeModel(model_name)

    payload = {
        "job_id": str(row["job_id"]),
        "company": str(row["company"]),
        "title_raw": str(row["title"]),
        "location_raw": str(row["location"]),
        "url": str(row["url"]),
        "raw_text": str(row["raw_text"]),
    }

    prompt = BASE_PROMPT + "\nINPUT JSON:\n" + json.dumps(payload)

    response = model.generate_content(prompt)
    text = response.text.strip()

    # Try to locate JSON in case of stray text
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)

    return json.loads(text)


## Validation and normalization


In [ ]:
def _ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [str(value)]

def _ensure_bool(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in {"true", "yes", "1"}
    if isinstance(value, (int, float)):
        return bool(value)
    return False

def _ensure_number_or_null(value):
    if value is None:
        return None
    try:
        num = float(value)
        if num.is_integer():
            return int(num)
        return num
    except Exception:
        return None

def normalize_record(rec):
    # Ensure all fields exist
    for f in SCHEMA_FIELDS:
        if f not in rec:
            rec[f] = None

    # Enforce allowed categories
    if rec["remote_type"] not in ALLOWED_REMOTE:
        rec["remote_type"] = "unknown"
    if rec["employment_type"] not in ALLOWED_EMPLOYMENT:
        rec["employment_type"] = "unknown"
    if rec["seniority"] not in ALLOWED_SENIORITY:
        rec["seniority"] = "unknown"

    # Lists
    rec["required_skills"] = _ensure_list(rec.get("required_skills"))
    rec["preferred_skills"] = _ensure_list(rec.get("preferred_skills"))
    rec["responsibilities"] = _ensure_list(rec.get("responsibilities"))
    rec["qualifications"] = _ensure_list(rec.get("qualifications"))

    # Booleans and numeric
    rec["salary_present"] = _ensure_bool(rec.get("salary_present"))
    rec["experience_years_min"] = _ensure_number_or_null(rec.get("experience_years_min"))
    rec["experience_years_max"] = _ensure_number_or_null(rec.get("experience_years_max"))

    # Ensure strings for key fields
    for k in ["job_id", "company", "title_raw", "location_raw", "url"]:
        rec[k] = "" if rec.get(k) is None else str(rec.get(k))

    return rec


## Run extraction on a small batch


In [ ]:
success_rows = []
failed_rows = []

for i, row in sample_10.iterrows():
    try:
        rec = extract_job_structured(row)
        rec = normalize_record(rec)
        success_rows.append(rec)
        print(f"[{len(success_rows)}] OK - {row['title']}")
    except Exception as e:
        failed_rows.append({
            "job_id": row.get("job_id"),
            "url": row.get("url"),
            "error": str(e)
        })
        print(f"[FAIL] {row['title']} -> {e}")

    time.sleep(1)  # gentle pacing


## Export outputs


In [ ]:
structured_df = pd.DataFrame(success_rows)
failed_df = pd.DataFrame(failed_rows)

structured_df.to_csv("structured_jobs.csv", index=False)
failed_df.to_csv("failed_jobs.csv", index=False)

print("Saved structured_jobs.csv and failed_jobs.csv")


## Quick analysis / query examples


In [ ]:
print("Total success:", len(structured_df))
print("Total failed:", len(failed_df))
print("Columns:", list(structured_df.columns))

display(structured_df.head(5))

# Count by remote_type
print(structured_df['remote_type'].value_counts(dropna=False))

# Count by employment_type
print(structured_df['employment_type'].value_counts(dropna=False))

# Rows where salary_present == True
print(structured_df[structured_df['salary_present'] == True].shape)

# Rows where required_skills contains 'python' (case-insensitive)
python_rows = structured_df[structured_df['required_skills'].apply(
    lambda xs: any('python' in str(x).lower() for x in xs)
)]
print(python_rows.shape)
